In [1]:
from shared.main import * 

<h1 align='center'> Find biologically plausible latent features</h1>

This is just a measurement model: descriptive 

The features should relate to: 
- within-participant
    - mapping score 
    - memory confusability 
    - RT 
- across-participant
    - avoidance

In [62]:
from typing import List, Dict, Any, Tuple

def trialwise_relationship(
    choices: np.ndarray,
    *,
    alpha: str | float = "1/t",
    eps: float = 1e-12,
    reference: tuple[float, float] | np.ndarray = (0.0, 0.0),
    center_for_update: bool = False,
) -> pd.DataFrame:
    """
    Trialwise, unified delta-rule features from choice vectors (returned as a DataFrame).

    Assumes D=2 with columns [affil, power].

    Definitions
    ----------
    Let c_t be the unit-normalized choice vector.

    - Mean (relationship location): m_t
        m_t = m_{t-1} + alpha_t * delta_t
    - Mismatch / prediction error: delta_t
        delta_t = c_t - m_{t-1}
    - Mean update (actual increment): update_t
        update_t = Δm_t = alpha_t * delta_t

    Reference point
    ---------------
    A reference point r (default (0,0)) is used only for *readouts* unless
    center_for_update=True.

    Readout magnitude:
        mean_magnitude_t = ||m_t - r||

    If center_for_update=True, then the state is tracked in centered coordinates:
        z_t = c_t - r
        ẑ_t updated by delta rule
        m_t reported back in original coordinates as ẑ_t + r

    Parameters
    ----------
    choices : (T, 2) array
        Trialwise choice vectors (e.g., [affil_choice, power_choice]).
        Will be unit-normalized per trial (eps-stabilized).
    alpha : "1/t" or float
        Learning rate. Default "1/t" gives exact running mean of unit vectors.
        If float in (0,1], uses constant learning rate (EWMA).
    eps : float
        Stabilizer for unit normalization.
    reference : tuple or (2,) array
        Reference point r = (r_affil, r_power) used for mean magnitude readout
        (and for centered updates if center_for_update=True).
    center_for_update : bool
        If False (default): update dynamics are unchanged; only mean_magnitude
        is computed relative to `reference`.
        If True: perform delta-rule updates on centered observations (c_t - r).

    Returns
    -------
    pd.DataFrame
        Columns:
          - decision_num: 1..T
          - affil_choice, power_choice: unit-normalized choice components
          - affil_mean, power_mean: running mean after update
          - affil_delta, power_delta: mismatch vector (c_t - m_{t-1}) in the
            coordinate system used for updating
          - affil_update, power_update: mean increment Δm_t = alpha_t * delta_t
          - alpha: learning rate per trial
          - mean_magnitude: ||m_t - reference||  (concentration around reference)
          - update_magnitude: ||Δm_t||
          - surprise: ||delta_t||^2
    """
    c = np.asarray(choices, dtype=float)
    if c.ndim != 2:
        raise ValueError(f"`choices` must be 2D (T,2); got shape {c.shape}")
    T, D = c.shape
    if D != 2:
        raise ValueError(f"`choices` must have D=2 columns [affil, power]; got D={D}")

    r = np.asarray(reference, dtype=float).reshape(-1)
    if r.shape != (2,):
        raise ValueError(f"`reference` must be length-2; got shape {r.shape}")

    # ---- unit-normalize each trial vector
    norms = np.linalg.norm(c, axis=1, keepdims=True)
    c_unit = c / (norms + eps)

    # If centering for update, operate on z_t = c_t - r; otherwise use c_t directly.
    obs = c_unit - r if center_for_update else c_unit

    # ---- allocate outputs
    m = np.zeros((T, D), dtype=float)         # mean in the *updating* coordinate system
    delta = np.zeros((T, D), dtype=float)     # obs_t - m_{t-1}
    update = np.zeros((T, D), dtype=float)    # Δm_t
    alpha_t = np.zeros((T,), dtype=float)

    m_prev = np.zeros((D,), dtype=float)

    for t in range(1, T + 1):
        # learning rate for this trial
        if isinstance(alpha, str):
            if alpha != "1/t":
                raise ValueError('If alpha is a string, it must be "1/t".')
            a = 1.0 / t
        else:
            a = float(alpha)
            if not (0.0 < a <= 1.0):
                raise ValueError(f"Constant alpha must be in (0,1]; got {a}")

        alpha_t[t - 1] = a

        dvec = obs[t - 1] - m_prev
        delta[t - 1] = dvec

        uvec = a * dvec
        update[t - 1] = uvec

        m_curr = m_prev + uvec
        m[t - 1] = m_curr
        m_prev = m_curr

    # Convert mean back to original coordinates for reporting if we centered updates.
    m_report = m + r if center_for_update else m

    # ---- derived scalars
    surprise = np.einsum("td,td->t", delta, delta)      # ||delta||^2 (in update coords)
    update_mag = np.linalg.norm(update, axis=1)         # ||Δm||

    # mean magnitude relative to reference (always computed in original coordinates)
    mean_mag = np.linalg.norm(m_report - r, axis=1)

    # ---- build dataframe
    # df = pd.DataFrame({"decision_num": np.arange(1, T + 1, dtype=int)})

    df = pd.DataFrame()

    # # vectors
    # df["affil_choice"] = c_unit[:, 0]
    # df["power_choice"] = c_unit[:, 1]

    df["affil_mean"] = m_report[:, 0]
    df["power_mean"] = m_report[:, 1]

    # delta/update are the quantities actually used in the delta-rule update (may be centered)
    df["affil_delta"] = delta[:, 0]
    df["power_delta"] = delta[:, 1]

    df["affil_update"] = update[:, 0]
    df["power_update"] = update[:, 1]

    # scalars
    df["mean_magnitude"] = mean_mag
    df["update_magnitude"] = update_mag
    df["surprise"] = surprise
    df["alpha"] = alpha_t

    return df

sub_id = 18001

data_row = data[data['sub_id'] == sub_id].iloc[0]

behavior = load_behavior(sub_id, neutrals=False)
choices = behavior[['affil_decision', 'power_decision']].to_numpy(dtype=float)
choices_by_char = organize_by_character(choices)
ests_by_char = [trialwise_relationship(choices_char) for choices_char in choices_by_char]
ests = organize_by_trial(ests_by_char)
ests = pd.concat([behavior, ests], axis=1)
ests.head()

,onset,decision_num,dimension,scene_num,character_role_num,character_decision_num,responded,reaction_time,affil_decision,power_decision,...,affil_mean,power_mean,affil_delta,power_delta,affil_update,power_update,mean_magnitude,update_magnitude,surprise,alpha
0,35.944,1,affil,1,1,1,True,6.051,-1,0,...,-1.00,0.00,-1.0,0.0,-1.00,0.00,1.000000,1.000000,1.0,1.000000
1,37.929,2,affil,1,1,2,True,2.616,-1,0,...,-1.00,0.00,0.0,0.0,0.00,0.00,1.000000,0.000000,0.0,0.500000
2,47.917,3,affil,1,1,3,True,6.664,-1,0,...,-1.00,0.00,0.0,0.0,0.00,0.00,1.000000,0.000000,0.0,0.333333
3,55.905,4,power,2,1,4,True,4.705,0,-1,...,-0.75,-0.25,1.0,-1.0,0.25,-0.25,0.790569,0.353553,2.0,0.250000
4,67.898,5,affil,2,2,1,True,6.064,1,0,...,1.00,0.00,1.0,0.0,1.00,0.00,1.000000,1.000000,1.0,1.000000


### features should re-capitulate previous findings

In [ ]:
from scipy.spatial.distance import pdist, squareform
from scipy.stats import kendalltau, rankdata, ttest_1samp, t

def compute_mapping_scores_fast(df, *, n_perm=1000, seed=2026):

    def mapping_score_distance_z_fast(beh, sub, perms):
        beh = np.asarray(beh, float)
        sub = np.asarray(sub, float)

        # cost matrix
        D = np.linalg.norm(beh[:, None, :] - sub[None, :, :], axis=2)
        obs = float(np.mean(np.diag(D)))

        null = D[np.arange(D.shape[0])[None, :], perms].mean(axis=1)
        sd = float(null.std(ddof=1))
        if sd == 0 or not np.isfinite(sd):
            return np.nan
        return float((null.mean() - obs) / sd)

    def _make_perms(n_char, n_perm, seed):
        rng = np.random.default_rng(seed)
        # sorting random noise → uniform random permutations
        return np.argsort(rng.random((n_perm, n_char)), axis=1)

    chars = list(CHARACTERS)
    n_char = len(chars)

    # (N, K, 2)
    beh = np.stack(
        [df[[f"affil_mean_{c}", f"power_mean_{c}"]].to_numpy(float) for c in chars],
        axis=1,
    )
    sub = np.stack(
        [df[[f"affil_subj_{c}", f"power_subj_{c}"]].to_numpy(float) for c in chars],
        axis=1,
    )

    perms = _make_perms(n_char, n_perm, seed)

    rows = []
    sids = df["sub_id"].to_numpy()
    for i, sid in enumerate(sids):
        rows.append(dict(
            sub_id=sid,
            mapping_distance_z=mapping_score_distance_z_fast(beh[i], sub[i], perms),
        ))

    return pd.DataFrame(rows)

def memory_confusions_by_subject_fast(data, *, normalize=True, symmetrize=True):
    chars = [str(c).lower() for c in CHARACTERS]
    K = len(chars)
    char_to_i = {c: i for i, c in enumerate(chars)}

    pat = re.compile(r"^memory_(\d{2})_(.+)$")
    mem = []
    for c in data.columns:
        m = pat.match(c)
        if not m:
            continue
        idx = int(m.group(1))
        if 1 <= idx <= 30:
            mem.append((idx, c, str(m.group(2)).lower()))

    if not mem:
        raise ValueError("No columns matching memory_01_* ... memory_30_* were found in `data`.")

    mem.sort()
    memory_cols = [c for _, c, _ in mem]
    true_labels = [t for _, _, t in mem]

    bad = [t for t in true_labels if t not in char_to_i]
    if bad:
        raise ValueError(f"Some true labels (from col suffix) not in CHARACTERS: {bad[:10]}")

    true_idx = np.array([char_to_i[t] for t in true_labels], dtype=int)

    # ---- responses: DataFrame -> object ndarray -> lowercase strings / None
    resp_obj = data.loc[:, memory_cols].to_numpy(dtype=object)

    # elementwise lower-casing; keep None for missing
    # (uses vectorized np.frompyfunc to avoid Python loops over subjects)
    lower = np.frompyfunc(lambda x: None if x is None or (isinstance(x, float) and np.isnan(x))
                          else str(x).lower(), 1, 1)
    resp_lower = lower(resp_obj)  # object array of lowercase strings or None

    # map to indices; unknowns -> -1
    resp_idx = np.full(resp_lower.shape, -1, dtype=int)
    for k, v in char_to_i.items():
        resp_idx[resp_lower == k] = v

    out = {}
    sub_ids = data["sub_id"].to_numpy()
    for r, sid in zip(resp_idx, sub_ids):
        ok = r >= 0
        flat = true_idx[ok] * K + r[ok]
        C = np.bincount(flat, minlength=K * K).reshape(K, K).astype(float)

        if normalize:
            rs = C.sum(axis=1, keepdims=True)
            C = np.divide(C, rs, out=np.zeros_like(C), where=rs > 0)

        if symmetrize:
            C = 0.5 * (C + C.T)

        out[sid] = C

    return out

def beh_rdm_by_subject(data):
    chars = list(CHARACTERS)
    rdm_by_sub = {}

    for i, sid in enumerate(data["sub_id"].to_numpy()):
        # (K, 2) in CHARACTERS order
        coords = np.array(
            [[data.iloc[i][f"affil_mean_{c}"], data.iloc[i][f"power_mean_{c}"]] for c in chars],
            dtype=float
        )
        rdm_by_sub[sid] = squareform(pdist(coords, metric="euclidean"))

    return rdm_by_sub

def subjectwise_partial_rsa_conf_vs_rdm(conf_by_sub, rdm_by_sub, onset_rdm):
    z_full = flatten_upper_tri(onset_rdm)

    rows = []
    for sid, conf in conf_by_sub.items():
        if sid not in rdm_by_sub:
            continue

        y_full = flatten_upper_tri(conf)
        x_full = flatten_upper_tri(rdm_by_sub[sid])

        ok = np.isfinite(y_full) & np.isfinite(x_full) & np.isfinite(z_full)
        n = int(ok.sum())
        if n < 3:
            rows.append(dict(sub_id=sid, r_partial=np.nan, n_pairs=n))
            continue

        yr = rankdata(y_full[ok], method="average")
        xr = rankdata(x_full[ok], method="average")
        zr = rankdata(z_full[ok], method="average")

        zr = zr - zr.mean()
        denom = float(np.dot(zr, zr))
        if denom <= 0 or not np.isfinite(denom):
            rows.append(dict(sub_id=sid, r_partial=np.nan, n_pairs=n))
            continue

        # residualize ranks on zr (no intercept needed after mean-centering zr)
        ry = yr - (np.dot(zr, yr) / denom) * zr
        rx = xr - (np.dot(zr, xr) / denom) * zr

        r = float(np.corrcoef(ry, rx)[0, 1])
        rows.append(dict(sub_id=sid, r_partial=r, n_pairs=n))

    return pd.DataFrame(rows)

def run_all_three_analyses_fast(
    *,
    data_inperson,
    data_online,
    decision_trials,
    n_perm_mapping=1000,
    seed_mapping=2026,
):
    
    def mean_onset_rdm(decision_trials):
        m = decision_trials.groupby("character_role_name")["onset"].mean()
        m = {str(k).lower(): float(v) for k, v in m.items()}
        onset = np.array([m.get(str(c).lower(), np.nan) for c in CHARACTERS], float)
        return np.abs(onset[:, None] - onset[None, :])

    def _summary_mean_t(x, popmean=0.0, alpha=0.05):
        x = np.asarray(x, float)
        x = x[np.isfinite(x)]
        n = x.size
        if n < 2:
            return dict(n=n, mean=np.nan, ci_lb=np.nan, ci_ub=np.nan, p=np.nan)

        m = float(x.mean())
        sd = float(x.std(ddof=1))
        sem = sd / np.sqrt(n)
        crit = float(t.ppf(1 - alpha / 2, n - 1))

        return dict(
            n=int(n),
            mean=m,
            ci_lb=m - crit * sem,
            ci_ub=m + crit * sem,
            p=float(ttest_1samp(x, popmean=popmean).pvalue),
        )
    
    def _summary_mean_r_fisher(r, popmean_r=0.0, alpha=0.05):
        r = np.asarray(r, float)
        r = r[np.isfinite(r)]
        r = np.clip(r, -0.999999, 0.999999)
        n = r.size
        if n < 2:
            return dict(n=n, mean=np.nan, ci_lb=np.nan, ci_ub=np.nan, p=np.nan)

        z = np.arctanh(r)
        z0 = float(np.arctanh(np.clip(popmean_r, -0.999999, 0.999999)))
        zm = float(z.mean())
        zs = float(z.std(ddof=1))
        zsem = zs / np.sqrt(n)
        crit = float(t.ppf(1 - alpha / 2, n - 1))

        return dict(
            n=int(n),
            mean=float(np.tanh(zm)),
            ci_lb=float(np.tanh(zm - crit * zsem)),
            ci_ub=float(np.tanh(zm + crit * zsem)),
            p=float(ttest_1samp(z, popmean=z0).pvalue),
        )

    def add(sample, analysis, term, s):
        out.append(dict(
            sample=sample,
            analysis=analysis,
            term=term,
            estimate=s["mean"],
            ci95_lb=s["ci_lb"],
            ci95_ub=s["ci_ub"],
            p=s["p"],
            n=s["n"],
        ))

    onset_rdm = mean_onset_rdm(decision_trials)
    out = []   

    # --- 1) Social avoidance regression (ONLY affil & power)
    for df, label in [(data_inperson, "in_person"), (data_online, "online")]:
        res, model = run_regression(
            "lsas_av_score ~ affil_mean_mean + power_mean_mean + memory_mean",
            data=df,
            return_obj=True,
        )
        nobs = int(getattr(model, "nobs", np.nan)) if model is not None else np.nan

        # keep ONLY these two
        keep_terms = {"affil_mean_mean", "power_mean_mean"}
        res_main = res[res["x"].isin(keep_terms)].copy()

        for _, r in res_main.iterrows():
            add(
                label,
                "social_avoidance",
                str(r["x"]),
                dict(
                    mean=float(r["beta"]),
                    ci_lb=float(r["95%_lb"]),
                    ci_ub=float(r["95%_ub"]),
                    p=float(r["p"]),
                    n=nobs,
                ),
            )

    # --- 2) Mapping scores
    for df, label in [(data_inperson, "in_person"), (data_online, "online")]:
        maps = compute_mapping_scores_fast(df, n_perm=n_perm_mapping, seed=seed_mapping)
        add(label, "mapping_scores", "mapping_distance_z",
            _summary_mean_t(maps["mapping_distance_z"]))


    # --- 3) Memory confusability partial RSA (term = single token; no wrap)
    for df, label in [(data_inperson, "in_person"), (data_online, "online")]:
        conf = memory_confusions_by_subject_fast(df, normalize=True, symmetrize=True)
        beh_rdm = beh_rdm_by_subject(df)
        part = subjectwise_partial_rsa_conf_vs_rdm(conf, beh_rdm, onset_rdm)

        add(
            label,
            "memory_confus",
            "confus_partial-time", 
            _summary_mean_r_fisher(part["r_partial"]),
        )

    return pd.DataFrame(out)

summary = run_all_three_analyses_fast(
    data_inperson=data,
    data_online=data_online,
    decision_trials=decision_trials,
    n_perm_mapping=50,
    seed_mapping=2026,
)
summary = summary.sort_values(["analysis", "sample", "term"]).reset_index(drop=True)
print_df(summary.round(3))

|    | sample    | analysis         | term                |   estimate |   ci95_lb |   ci95_ub |     p |   n |
|---:|:----------|:-----------------|:--------------------|-----------:|----------:|----------:|------:|----:|
|  0 | in_person | mapping_scores   | mapping_distance_z  |      0.519 |     0.275 |     0.764 | 0     |  74 |
|  1 | online    | mapping_scores   | mapping_distance_z  |      0.728 |     0.657 |     0.8   | 0     | 791 |
|  2 | in_person | memory_confus    | confus_partial-time |     -0.057 |    -0.12  |     0.007 | 0.078 |  79 |
|  3 | online    | memory_confus    | confus_partial-time |     -0.056 |    -0.076 |    -0.036 | 0     | 779 |
|  4 | in_person | social_avoidance | affil_mean_mean     |     -1.788 |    -5.23  |     1.653 | 0.304 |  79 |
|  5 | in_person | social_avoidance | power_mean_mean     |      1.396 |    -2.445 |     5.237 | 0.471 |  79 |
|  6 | online    | social_avoidance | affil_mean_mean     |     -2.413 |    -3.527 |    -1.299 | 0     | 790 |
|  7 | online    | social_avoidance | power_mean_mean     |      1.854 |     0.728 |     2.98  | 0.001 | 790 |